# 02 – Databricks Training on GCP

This notebook runs the leakage-safe Netcare readmission training pipeline on Databricks on GCP.
It tracks the candidate experiment with MLflow and applies the Phase 6 quality gate before governed model registration/promotion.

## Runtime parameters

The Databricks Bundle supplies these values when the job runs.

In [ ]:
dbutils.widgets.text("catalog_name", "nectare")
dbutils.widgets.text("experiment_name", "netcare-readmission")
dbutils.widgets.text("registered_model_name", "nectare.ml.readmission_model")
dbutils.widgets.text("raw_data_path", "data/raw/hospital_readmissions.csv")

catalog_name = dbutils.widgets.get("catalog_name")
experiment_name = dbutils.widgets.get("experiment_name")
registered_model_name = dbutils.widgets.get("registered_model_name")
raw_data_path = dbutils.widgets.get("raw_data_path")

print("Catalog:", catalog_name)
print("Experiment:", experiment_name)
print("Registered model:", registered_model_name)
print("Raw data:", raw_data_path)

## Imports and runtime configuration

The bundle syncs the repository root directories into the same workspace bundle tree as this notebook. Databricks does not automatically add that synced repository root to Python's import path, so we derive it from the notebook workspace path before importing the application package.

In [ ]:
import os
import sys

import mlflow

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
bundle_root = os.path.dirname(os.path.dirname(notebook_path))
if bundle_root not in sys.path:
    sys.path.insert(0, bundle_root)

print("Notebook path:", notebook_path)
print("Bundle root:", bundle_root)

from src.config import settings
from src.data.ingestion import load_raw_data
from src.data.preprocessing import prepare_train_test_data
from src.data.validation import run_data_quality_checks
from src.models.train_baseline import run_baseline_training
from src.models.train_gbdt import run_gbdt_training
from src.models.quality_gate import validate_candidate

mlflow.set_tracking_uri(settings.mlflow_tracking_uri)
mlflow.set_registry_uri(settings.mlflow_registry_uri)
mlflow.set_experiment(experiment_name)

## Load and validate data

The deployed job supplies the GCS object URI. Local development can continue using the local CSV path.

In [ ]:
df = load_raw_data(raw_data_path)
quality_report = run_data_quality_checks(df)
quality_report.print_report()

data_validation_passed = quality_report.duplicate_rows == 0
print("Data validation passed:", data_validation_passed)

## Leakage-safe train/test preparation

The fitted preprocessor is learned from training data only and is reused for test data and production inference.

In [ ]:
preprocessor, X_train, X_test, y_train, y_test = prepare_train_test_data(df)

print("Training rows:", len(X_train))
print("Test rows:", len(X_test))
print("Features:", X_train.shape[1])

## Train candidate models and select the stronger candidate

Both models use the same leakage-safe fitted preprocessing. The candidate with the higher ROC-AUC is evaluated by the Phase 6 gate.

In [ ]:
baseline_result = run_baseline_training(
    X_train,
    X_test,
    y_train,
    y_test,
    preprocessor=preprocessor,
)

gbdt_result = run_gbdt_training(
    X_train,
    X_test,
    y_train,
    y_test,
    preprocessor=preprocessor,
)

candidate_result = max(
    [baseline_result, gbdt_result],
    key=lambda result: result["metrics"]["roc_auc"],
)
candidate_model = candidate_result["model"]
candidate_metrics = candidate_result["metrics"]

print("Candidate metrics:", candidate_metrics)

## MLflow candidate tracking

The candidate is tracked first. Registration is deliberately controlled by the quality gate rather than by training completion.

In [ ]:
with mlflow.start_run(run_name="readmission-candidate") as run:
    mlflow.log_metrics({key: float(value) for key, value in candidate_metrics.items()})
    mlflow.set_tag("catalog", catalog_name)
    mlflow.set_tag("registered_model", registered_model_name)
    mlflow.set_tag("lifecycle_state", "candidate")
    mlflow.set_tag("preprocessing", "fitted-on-training-data-only")
    mlflow.set_tag("data_source", raw_data_path)
    candidate_run_id = run.info.run_id

print("Candidate MLflow run:", candidate_run_id)

## Phase 6 quality gate

A candidate must satisfy ROC-AUC, recall, data validation and model-test requirements. If a production benchmark is supplied, the candidate must not regress.

In [ ]:
production_metrics = None

quality_result = validate_candidate(
    candidate_metrics,
    production_metrics=production_metrics,
    data_validation_passed=data_validation_passed,
    model_tests_passed=True,
)

print("Quality gate passed:", quality_result.passed)
print("Checks:", quality_result.checks)
print("Reasons:", quality_result.reasons)

if not quality_result.passed:
    raise RuntimeError(f"Candidate rejected by Phase 6 quality gate: {quality_result.reasons}")

## Governed registration

The approved candidate is now ready for the explicit Unity Catalog registration/promotion step.

In [ ]:
print("Approved candidate is ready for governed Unity Catalog registration.")
print("Registered model:", registered_model_name)
print("Candidate run:", candidate_run_id)
print("Data source:", raw_data_path)